# Databricks Workspace Discovery Scanner

This notebook performs an automated discovery of Databricks workspace assets, configurations, and usage patterns.  
It is designed to help administrators gain visibility into workspace structure, Cluster Information, Job Details, Pipelines, unity catalog, security information, utilization metrics , and operational artifacts.


### Key Outcomes
- Workspace-wide inventory of compute, jobs, pipelines, and governance objects
- Utilization and cost-related signals to support optimization discussions
- Structured outputs for cataloging of Databricks Assets in a workspace

## Environment Setup and Toolkit Installation

This cell installs and initializes the required Databricks Toolkit components used for workspace discovery.
It ensures that all supporting libraries and dependencies are consistently available before execution begins.

**Why this matters**
- Guarantees version-aligned tooling
- Avoids dependency conflicts during scan execution
- Enables repeatable execution across workspaces and environments


In [0]:
# Replace the path with your wheel file location
%pip install --force-reinstall --no-deps --no-cache-dir /Volumes/rkdbscannercatalog/default/rkdbscannervolumeout/script/databricks_toolkit-2.0.2-py3-none-any.whl

Processing /Volumes/rkdbscannercatalog/default/rkdbscannervolumeout/script/databricks_toolkit-2.0.2-py3-none-any.whl
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Python Runtime Restart

The Python runtime is restarted to activate newly installed packages.
This step ensures that all imports and toolkit modules are available to subsequent cells.

> Note: This operation clears in-memory state and should only be run after dependency installation.


In [0]:
# Restart Python to load the newly installed package
dbutils.library.restartPython()

## Scan Configuration

This section defines all configurable parameters that control the discovery scan behavior.

### Key Settings
- **Workspace URL** – Target Databricks workspace
- **Scan Type** – Simple or Deep (controls metadata vs. utilization depth)
- **Unity Catalog Mode** – Partial or Complete hierarchy scan
- **Billing & Utilization Windows** – Defines historical analysis range
- **Parallelism Controls** – Worker concurrency to balance performance and load

These parameters should be reviewed and customized per environment before running the scan.

In [0]:
from databricks_toolkit.discovery import ScanConfig, WorkspaceDiscoveryScanner

# ===================================================================
# CONFIGURATION - Update these values for your environment
# ===================================================================

# Your Databricks workspace URL
WORKSPACE_URL = "https://adb-7405617048577650.10.azuredatabricks.net/"

# Output directory where discovery data will be saved
# This data will be used by the analysis notebook
DISCOVERY_OUTPUT = "/Volumes/rkdbscannercatalog/default/rkdbscannervolumeout/output/discovery/"

# Unity Catalog target for scan results
CATALOG_NAME = "dbscannercatalog"        # Target catalog name
SCHEMA_NAME  = "catalog_inventory"      # Target schema name

# Scan configuration
SCAN_TYPE = 'deep'              # 'simple' or 'deep' (deep collects utilization metrics)
UC_SCAN = 'partial'             # 'partial' (faster, tables only) or 'complete' (full hierarchy)
BILLING_ENABLED = 'Y'           # 'Y' to collect billing data, 'N' to skip
BILLING_DAYS = 30               # Number of days of billing data to collect (1-30)
UTILIZATION_DAYS = 180          # Number of days of utilization metrics (for deep scan)

# Performance settings
MAX_WORKERS = 6                 # Parallel threads (1-10, higher = faster but more load)
EXPORT_FORMAT = 'json'          # 'json' or 'csv'

print("✓ Configuration loaded")
print(f"  Workspace: {WORKSPACE_URL}")
print(f"  Output: {DISCOVERY_OUTPUT}")
print(f"  Catalog: {CATALOG_NAME}.{SCHEMA_NAME}")
print(f"  Scan Type: {SCAN_TYPE}")
print(f"  UC Scan Mode: {UC_SCAN}")


✓ Configuration loaded
  Workspace: https://adb-7405617048577650.10.azuredatabricks.net/
  Output: /Volumes/rkdbscannercatalog/default/rkdbscannervolumeout/output/discovery/
  Scan Type: deep
  UC Scan Mode: partial


## Run Discovery Scan
This initializes the workspace discovery scanner using the provided configuration and save results to the specified output directory.

In [0]:
# Create scan configuration
config = ScanConfig(
    scan_type=SCAN_TYPE,
    workspace_urls=[WORKSPACE_URL],
    uc_scan=UC_SCAN,
    billing_details=BILLING_ENABLED,
    billing_days=BILLING_DAYS,
    utilization_days=UTILIZATION_DAYS,
    export_format=EXPORT_FORMAT,
    max_workers=MAX_WORKERS,
    output_file=DISCOVERY_OUTPUT,
    account_id = 'b34ffcd1-4a03-4d3c-8b82-c99e27dd6924',
    log_level='INFO'
)

print("="*80)
print("STARTING WORKSPACE DISCOVERY SCAN")
print("="*80)
print(f"\nScan Configuration:")
print(f"  • Scan Type: {config.scan_type.upper()}")
print(f"  • UC Scan: {config.uc_scan.upper()}")
print(f"  • Billing: {'Enabled' if config.is_billing_enabled() else 'Disabled'} ({config.billing_days} days)")
print(f"  • Utilization: {config.utilization_days} days")
print(f"  • Workers: {config.max_workers}")
print(f"  • Format: {config.export_format.upper()}")
print(f"\n" + "="*80 + "\n")

# Initialize scanner and run
scanner = WorkspaceDiscoveryScanner(config)
results = scanner.scan_workspaces()

print(f"\n" + "="*80)
print("✓ DISCOVERY SCAN COMPLETE")
print("="*80)

2026-02-05 15:14:08,111 - py4j.clientserver - INFO - Received command c on object id p0
STARTING WORKSPACE DISCOVERY SCAN

Scan Configuration:
  • Scan Type: DEEP
  • UC Scan: PARTIAL
  • Billing: Enabled (30 days)
  • Utilization: 180 days
  • Workers: 6
  • Format: JSON


2026-02-05 15:14:08,117 - databricks_toolkit.discovery.main_scanner - INFO - Starting workspace discovery scan for 1 workspace(s)
2026-02-05 15:14:08,119 - databricks_toolkit.discovery.main_scanner - INFO - 
Scanning workspace: https://adb-7405617048577650.10.azuredatabricks.net/
2026-02-05 15:14:08,119 - databricks_toolkit.discovery.main_scanner - INFO - Authenticating to workspace...
2026-02-05 15:14:08,120 - databricks_toolkit.discovery.auth - INFO - Creating new client for workspace: https://adb-7405617048577650.10.azuredatabricks.net/
2026-02-05 15:14:08,298 - databricks_toolkit.discovery.auth - INFO - Successfully authenticated to https://adb-7405617048577650.10.azuredatabricks.net/ as: admin@mngenvmcap761126.

## Scan Results Summary Catalog Dashboards

This section summarizes the overall completion of the discovery scan.
It reports execution duration, scan tier performance, and artifact coverage.

✅ Successful completion indicates:
- All configured scan tiers executed
- Results available for analysis and populated in Dashboard
- Output artifacts written to storage

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import to_timestamp, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, TimestampType

import uuid

# ── Table references (resolved from config) ─────────────────────────────────
TBL_SUMMARY    = f"{CATALOG_NAME}.{SCHEMA_NAME}.workspace_scan_summary"
TBL_CLUSTERS   = f"{CATALOG_NAME}.{SCHEMA_NAME}.workspace_scan_clusters"
TBL_UC_TABLES  = f"{CATALOG_NAME}.{SCHEMA_NAME}.workspace_scan_uc_tables"
TBL_WAREHOUSES = f"{CATALOG_NAME}.{SCHEMA_NAME}.workspace_scan_warehouses"
TBL_PIPELINES  = f"{CATALOG_NAME}.{SCHEMA_NAME}.workspace_scan_pipelines"

print(f"  Target tables: {CATALOG_NAME}.{SCHEMA_NAME}.*")

# Helper to extract artifact counts safely
def get_count(artifacts, key, default=0):
    val = artifacts.get(key, {})
    if isinstance(val, dict):
        return int(val.get('total_count', default))
    elif isinstance(val, list):
        return int(len(val))
    return int(default)

# Helper to safely get nested dict values
def get_nested(artifacts, key, subkey, default=0):
    val = artifacts.get(key, {})
    if isinstance(val, dict):
        subval = val.get(subkey, default)
        # Handle case where subval might be a list
        if isinstance(subval, list):
            return int(len(subval))
        return int(subval)
    return int(default)

# Prepare rows for DataFrame
cur_scan_id=str(uuid.uuid4())
rows = []
for ws in results.get('workspaces', []):
    artifacts = ws.get('artifacts', {})
    row = Row(
        scan_id=cur_scan_id,
        workspace_id=ws.get('workspace_id', ''),
        workspace_url=ws.get('workspace_url', ''),
        scan_start_timestamp=results['scan_metadata'].get('start_time', ''),
        scan_end_timestamp=results['scan_metadata'].get('end_time', ''),
        scan_duration_seconds=float(results['scan_metadata'].get('total_duration_seconds', 0.0)),
        scan_type=results['scan_metadata'].get('scan_type', ''),
        total_clusters=get_count(artifacts, 'clusters'),
        total_jobs=get_count(artifacts, 'jobs'),
        total_pipelines=get_count(artifacts, 'pipelines'),
        total_uc_tables=int(artifacts.get('unity_catalog', {}).get('summary', {}).get('total_tables', 0)),
        total_warehouses=get_count(artifacts, 'warehouses'),
        total_ml_experiments=get_nested(artifacts, 'workspace_objects', 'experiments'),
        total_repos=get_nested(artifacts, 'workspace_objects', 'repos'),
        total_serving_endpoints=get_nested(artifacts, 'workspace_objects', 'serving_endpoints'),
        total_genie_spaces=get_nested(artifacts, 'workspace_objects', 'genie_spaces'),
        total_alerts=get_count(artifacts, 'alerts'),
        total_notebooks=get_nested(artifacts, 'workspace_objects', 'notebooks')
    )
    rows.append(row)

df = spark.createDataFrame(rows)

# Convert scan_timestamp string to timestamp type
df = df.withColumn('scan_start_timestamp', to_timestamp('scan_start_timestamp'))
df = df.withColumn('scan_end_timestamp', to_timestamp('scan_end_timestamp'))

# Cast all integer columns to int type to match table schema
int_columns = [
    'total_clusters', 'total_jobs', 'total_pipelines', 'total_uc_tables', 'total_warehouses',
    'total_ml_experiments', 'total_repos','total_serving_endpoints', 'total_genie_spaces', 'total_alerts', 'total_notebooks'
]

for col_name in int_columns:
    df = df.withColumn(col_name, col(col_name).cast('int'))

df.write.mode("append").saveAsTable(TBL_SUMMARY)

cluster_rows = []
for ws in results.get('workspaces', []):
    artifacts = ws.get('artifacts', {})
    clusters = artifacts.get('clusters', {}).get('items', []) if isinstance(artifacts.get('clusters', {}), dict) else artifacts.get('clusters', [])
    for c in clusters:
        cluster_rows.append(Row(
            workspace_id=ws.get('workspace_id', ''),
            scan_id=cur_scan_id,
            cluster_id=c.get('cluster_id', ''),
            cluster_name=c.get('cluster_name', ''),
            state=c.get('state', ''),
            cluster_source=c.get('cluster_source', ''),
            node_type_id=c.get('node_type_id', ''),
            driver_node_type_id=c.get('driver_node_type_id', ''),
            min_workers=c.get('min_workers'),
            max_workers=c.get('max_workers'),
            num_workers=c.get('num_workers'),
            autotermination_minutes=c.get('autotermination_minutes'),
            creator_user_name=c.get('creator_user_name', ''),
            uptime_hours=c.get('uptime_hours'),
            spot_enabled=c.get('spot', c.get('spot_enabled', None)),
            enable_elastic_disk=c.get('enable_elastic_disk'),
            spark_version=c.get('spark_version', ''),
            policy_id=c.get('policy_id', '')
        ))

cluster_schema = StructType([
    StructField("workspace_id", StringType(), False),
    StructField("scan_id", StringType(), False),
    StructField("cluster_id", StringType(), False),
    StructField("cluster_name", StringType(), True),
    StructField("state", StringType(), True),
    StructField("cluster_source", StringType(), True),
    StructField("node_type_id", StringType(), True),
    StructField("driver_node_type_id", StringType(), True),
    StructField("min_workers", IntegerType(), True),
    StructField("max_workers", IntegerType(), True),
    StructField("num_workers", IntegerType(), True),
    StructField("autotermination_minutes", IntegerType(), True),
    StructField("creator_user_name", StringType(), True),
    StructField("uptime_hours", DoubleType(), True),
    StructField("spot_enabled", BooleanType(), True),
    StructField("enable_elastic_disk", BooleanType(), True),
    StructField("spark_version", StringType(), True),
    StructField("policy_id", StringType(), True)
])

cluster_df = spark.createDataFrame(cluster_rows, schema=cluster_schema)
cluster_df.write.mode("append").saveAsTable(TBL_CLUSTERS)

uc_table_rows = []
for ws in results.get('workspaces', []):
    artifacts = ws.get('artifacts', {})
    uc = artifacts.get('unity_catalog', {})
    tables = []
    # Try summary->tables, else fallback to tables key
    if isinstance(uc, dict):
        if 'tables' in uc:
            tables = uc['tables']
        elif 'summary' in uc and 'tables' in uc['summary']:
            tables = uc['summary']['tables']
    for t in tables:
        uc_table_rows.append(Row(
            workspace_id=ws.get('workspace_id', ''),
            scan_id=cur_scan_id,
            catalog_name=t.get('catalog_name', ''),
            schema_name=t.get('schema_name', ''),
            table_name=t.get('table_name', ''),
            table_type=t.get('table_type', ''),
            table_format=t.get('data_source_format') if t.get('data_source_format') is not None else '',
            storage_location=t.get('metastore_path') if t.get('data_source_format') is not None else '',
            is_delta=bool(t.get('is_delta', False))
        ))

uc_table_schema = StructType([
    StructField("workspace_id", StringType(), False),
    StructField("scan_id", StringType(), False),
    StructField("catalog_name", StringType(), True),
    StructField("schema_name", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("table_type", StringType(), True),
    StructField("table_format", StringType(), True),
    StructField("storage_location", StringType(), True),
    StructField("is_delta", BooleanType(), True)
])

workspace_scan_uc_tables_df = spark.createDataFrame(uc_table_rows, schema=uc_table_schema)
workspace_scan_uc_tables_df.write.mode("append").saveAsTable(TBL_UC_TABLES)

warehouse_rows = []
for ws in results.get('workspaces', []):
    artifacts = ws.get('artifacts', {})
    warehouses = artifacts.get('warehouses', {}).get('items', []) if isinstance(artifacts.get('warehouses', {}), dict) else artifacts.get('warehouses', [])
    for w in warehouses:
        warehouse_rows.append(Row(
            workspace_id=ws.get('workspace_id', ''),
            scan_id=cur_scan_id,
            warehouse_id=w.get('warehouse_id', ''),
            warehouse_name=w.get('warehouse_name', ''),
            cluster_size=w.get('cluster_size', ''),
            min_num_clusters=w.get('min_num_clusters'),
            max_num_clusters=w.get('max_num_clusters'),
            auto_stop_mins=w.get('auto_stop_mins'),
            state=w.get('state', ''),
            warehouse_type=w.get('warehouse_type', ''),
            enable_photon=w.get('enable_photon'),
            enable_serverless_compute=w.get('enable_serverless_compute'),
            spot_instance_policy=w.get('spot_instance_policy', ''),
            creator_name=w.get('creator_name', '')
        ))

warehouse_schema = StructType([
    StructField("workspace_id", StringType(), True),
    StructField("scan_id", StringType(), False),
    StructField("warehouse_id", StringType(), False),
    StructField("warehouse_name", StringType(), True),
    StructField("cluster_size", StringType(), True),
    StructField("min_num_clusters", IntegerType(), True),
    StructField("max_num_clusters", IntegerType(), True),
    StructField("auto_stop_mins", IntegerType(), True),
    StructField("state", StringType(), True),
    StructField("warehouse_type", StringType(), True),
    StructField("enable_photon", BooleanType(), True),
    StructField("enable_serverless_compute", BooleanType(), True),
    StructField("spot_instance_policy", StringType(), True),
    StructField("creator_name", StringType(), True)
])

warehouse_df = spark.createDataFrame(warehouse_rows, schema=warehouse_schema)
warehouse_df.write.mode("append").saveAsTable(TBL_WAREHOUSES)

pipeline_rows = []
for ws in results.get('workspaces', []):
    artifacts = ws.get('artifacts', {})
    pipelines = artifacts.get('pipelines', {}).get('items', []) if isinstance(artifacts.get('pipelines', {}), dict) else artifacts.get('pipelines', [])
    for p in pipelines:
        pipeline_rows.append(Row(
            workspace_id=ws.get('workspace_id', ''),
            scan_id=cur_scan_id,
            pipeline_id=p.get('pipeline_id', ''),
            pipeline_name=p.get('name', ''),
            state=p.get('state', ''),
            creator_user_name=p.get('creator_user_name', ''),
            creation_time=p.get('creation_time', None)
        ))

pipeline_schema = StructType([
    StructField("workspace_id", StringType(), False),
    StructField("scan_id", StringType(), False),
    StructField("pipeline_id", StringType(), False),
    StructField("pipeline_name", StringType(), True),
    StructField("state", StringType(), True),
    StructField("creator_user_name", StringType(), True),
    StructField("creation_time", TimestampType(), True)
])

pipeline_df = spark.createDataFrame(pipeline_rows, schema=pipeline_schema)
pipeline_df = pipeline_df.withColumn("creation_time", to_timestamp("creation_time"))
pipeline_df.write.mode("append").saveAsTable(TBL_PIPELINES)


2026-02-05 15:15:23,929 - py4j.clientserver - INFO - Received command c on object id p0
2026-02-05 15:15:24,929 - py4j.clientserver - INFO - Received command c on object id p0
2026-02-05 15:15:25,929 - py4j.clientserver - INFO - Received command c on object id p0
2026-02-05 15:15:26,929 - py4j.clientserver - INFO - Received command c on object id p0
2026-02-05 15:15:27,929 - py4j.clientserver - INFO - Received command c on object id p0
2026-02-05 15:15:28,929 - py4j.clientserver - INFO - Received command c on object id p0
2026-02-05 15:15:29,929 - py4j.clientserver - INFO - Received command c on object id p0
2026-02-05 15:15:30,929 - py4j.clientserver - INFO - Received command c on object id p0
2026-02-05 15:15:31,929 - py4j.clientserver - INFO - Received command c on object id p0
